# [Laptop Price Prediction](https://www.kaggle.com/datasets/eslamelsolya/laptop-price-prediction)

Целью этого проекта является прогнозирование цен на ноутбуки на основе их технических характеристик, таких как тип процессора, объем оперативной памяти, емкость накопителя, марка и другие ключевые характеристики. Используя регрессионную модель, проект анализирует взаимосвязь между этими характеристиками и соответствующими ценами на ноутбуки, чтобы получить точные прогнозы цен.

### Признаки

* **неназванный признак** - ничего не известно
* **Company** компания-производитель
* **TypeName** тип ноутбука
* **Inches** размер диагонали матрицы в дюймах
* **ScreenResolution** тип матрицы и разрешение экрана
* **Cpu** тип процессора
* **Ram** размер оперативной памяти
* **Memory** размер и тип памяти
* **Gpu** видеокарта
* **OpSys** операционная система
* **Weight** вес устройства
* **Price** цена - целевой признак

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rc('font', family='Verdana', size=16)

import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
laptop = pd.read_csv('laptop_data.csv',sep = ',', header = 0)
laptop.columns

Index(['Unnamed: 0', 'Company', 'TypeName', 'Inches', 'ScreenResolution',
       'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price'],
      dtype='object')

Удаляем бесполезный `Unnamed: 0`

In [ ]:
laptop.drop(['Unnamed: 0'], axis = 1, inplace = True)
laptop

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080
...,...,...,...,...,...,...,...,...,...,...,...
1298,Lenovo,2 in 1 Convertible,14.0,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 6500U 2.5GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.8kg,33992.6400
1299,Lenovo,2 in 1 Convertible,13.3,IPS Panel Quad HD+ / Touchscreen 3200x1800,Intel Core i7 6500U 2.5GHz,16GB,512GB SSD,Intel HD Graphics 520,Windows 10,1.3kg,79866.7200
1300,Lenovo,Notebook,14.0,1366x768,Intel Celeron Dual Core N3050 1.6GHz,2GB,64GB Flash Storage,Intel HD Graphics,Windows 10,1.5kg,12201.1200
1301,HP,Notebook,15.6,1366x768,Intel Core i7 6500U 2.5GHz,6GB,1TB HDD,AMD Radeon R5 M330,Windows 10,2.19kg,40705.9200


**Целевая переменная ($y$)** - `Price`
_____

In [ ]:
laptop.describe()

,Inches,Price
count,1303.000000,1303.000000
mean,15.017191,59870.042910
std,1.426304,37243.201786
min,10.100000,9270.720000
25%,14.000000,31914.720000
50%,15.600000,52054.560000
75%,15.600000,79274.246400
max,18.400000,324954.720000


In [ ]:
laptop.describe(include=['object', 'category'])

,Company,TypeName,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
count,1303,1303,1303,1303,1303,1303,1303,1303,1303
unique,19,6,40,118,9,39,110,9,179
top,Dell,Notebook,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.2kg
freq,297,727,507,190,619,412,281,1072,121


In [ ]:
laptop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           1303 non-null   object 
 1   TypeName          1303 non-null   object 
 2   Inches            1303 non-null   float64
 3   ScreenResolution  1303 non-null   object 
 4   Cpu               1303 non-null   object 
 5   Ram               1303 non-null   object 
 6   Memory            1303 non-null   object 
 7   Gpu               1303 non-null   object 
 8   OpSys             1303 non-null   object 
 9   Weight            1303 non-null   object 
 10  Price             1303 non-null   float64
dtypes: float64(2), object(9)
memory usage: 112.1+ KB


## Признак `ScreenResolution` разбить на 3:
- **Screen_type**: Тип экрана (Full HD, 4K Ultra HD, Nan и т.д.)
- **SR_weight**: Разрешение - ширина (извлекать из 2560x1600 и т.д. корректно ширину - 2560)
- **SR_height**: Разрешение - высота (извлекать из 2560x1600 и т.д. корректно высоту - 1600)

Подумать как это сделать.

In [ ]:
laptop.ScreenResolution.unique()

array(['IPS Panel Retina Display 2560x1600', '1440x900',
       'Full HD 1920x1080', 'IPS Panel Retina Display 2880x1800',
       '1366x768', 'IPS Panel Full HD 1920x1080',
       'IPS Panel Retina Display 2304x1440',
       'IPS Panel Full HD / Touchscreen 1920x1080',
       'Full HD / Touchscreen 1920x1080',
       'Touchscreen / Quad HD+ 3200x1800',
       'IPS Panel Touchscreen 1920x1200', 'Touchscreen 2256x1504',
       'Quad HD+ / Touchscreen 3200x1800', 'IPS Panel 1366x768',
       'IPS Panel 4K Ultra HD / Touchscreen 3840x2160',
       'IPS Panel Full HD 2160x1440',
       '4K Ultra HD / Touchscreen 3840x2160', 'Touchscreen 2560x1440',
       '1600x900', 'IPS Panel 4K Ultra HD 3840x2160',
       '4K Ultra HD 3840x2160', 'Touchscreen 1366x768',
       'IPS Panel Full HD 1366x768', 'IPS Panel 2560x1440',
       'IPS Panel Full HD 2560x1440',
       'IPS Panel Retina Display 2736x1824', 'Touchscreen 2400x1600',
       '2560x1440', 'IPS Panel Quad HD+ 2560x1440',
       'IPS Panel 

In [ ]:
#laptop[['Screen_type', 'SR_weight', 'SR_height']] = laptop['ScreenResolution'].str.extract(r'([A-Za-z\d/\s]*)\s(\d{,4})x(\d{,4})')
laptop[['Screen_type', 'SR_weight', 'SR_height']] = laptop['ScreenResolution'].str.extract(r'^(.*?)\s*(\d{,4})x(\d{,4})')
#laptop[['Screen_type', 'SR_weight', 'SR_height']] = laptop['ScreenResolution'].str.extract(r'([\d\D]*)\s(\d{,4})x(\d{,4})')

In [ ]:
laptop['Screen_type'].unique()

array(['IPS Panel Retina Display', '', 'Full HD', 'IPS Panel Full HD',
       'IPS Panel Full HD / Touchscreen', 'Full HD / Touchscreen',
       'Touchscreen / Quad HD+', 'IPS Panel Touchscreen', 'Touchscreen',
       'Quad HD+ / Touchscreen', 'IPS Panel',
       'IPS Panel 4K Ultra HD / Touchscreen', '4K Ultra HD / Touchscreen',
       'IPS Panel 4K Ultra HD', '4K Ultra HD', 'IPS Panel Quad HD+',
       'IPS Panel Quad HD+ / Touchscreen',
       'IPS Panel Touchscreen / 4K Ultra HD', 'Touchscreen / Full HD',
       'Quad HD+', 'Touchscreen / 4K Ultra HD'], dtype=object)

In [ ]:
laptop['SR_weight'] = laptop['SR_weight'].astype(int)
laptop['SR_weight'].unique()

array([2560, 1440, 1920, 2880, 1366, 2304, 3200, 2256, 3840, 2160, 1600,
       2736, 2400])

In [ ]:
laptop['SR_height'] = laptop['SR_height'].astype(int)
laptop['SR_height'].unique()

array([1600,  900, 1080, 1800,  768, 1440, 1200, 1504, 2160, 1824])

In [ ]:
laptop[laptop['Screen_type'].str.contains("Touchscreen")]['Screen_type']

,Screen_type
19,IPS Panel Full HD / Touchscreen
23,Full HD / Touchscreen
33,Touchscreen / Quad HD+
44,Full HD / Touchscreen
50,IPS Panel Touchscreen
...,...
1271,IPS Panel Quad HD+ / Touchscreen
1284,IPS Panel Full HD / Touchscreen
1285,IPS Panel Quad HD+ / Touchscreen
1298,IPS Panel Full HD / Touchscreen


Почему бы не создать отдельный столбец с тачскрином

In [ ]:
laptop['Touchscreen'] = laptop['Screen_type'].str.contains('Touchscreen').astype(int)
laptop['Touchscreen']

,Touchscreen
0,0
1,0
2,0
3,0
4,0
...,...
1298,1
1299,1
1300,0
1301,0


In [ ]:
laptop['Screen_type'] = laptop['Screen_type'].str.extract(r'(\w+\s+HD)')
laptop['Screen_type'].unique()

array([nan, 'Full HD', 'Quad HD', 'Ultra HD'], dtype=object)

в Screen_type есть пропуски, но параметры экрана нам всегда известны, поэтому можно заполнить признак по ним

In [ ]:
laptop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           1303 non-null   object 
 1   TypeName          1303 non-null   object 
 2   Inches            1303 non-null   float64
 3   ScreenResolution  1303 non-null   object 
 4   Cpu               1303 non-null   object 
 5   Ram               1303 non-null   object 
 6   Memory            1303 non-null   object 
 7   Gpu               1303 non-null   object 
 8   OpSys             1303 non-null   object 
 9   Weight            1303 non-null   object 
 10  Price             1303 non-null   float64
 11  Screen_type       916 non-null    object 
 12  SR_weight         1303 non-null   int64  
 13  SR_height         1303 non-null   int64  
 14  Touchscreen       1303 non-null   int64  
dtypes: float64(2), int64(3), object(10)
memory usage: 152.8+ KB


In [ ]:
mpx_to_type = { #https://en.wikipedia.org/wiki/Display_resolution
    0.922: "SWXGA",
    1.024: "WXGA",
    1.311: "SXGA",
    1.044: "HD",
    1.049: "HD",
    1.296: "WXGA+",
    1.440: "HD+",
    1.920: "UXGA",
    1.764: "NWSXGA+",
    2.074: "FHD",
    2.304: "WUXGA",
    2.765: "UWFHD",
    3.686: "QHD",
    4.096: "WQXGA",
    4.954: "UWQHD",
    8.294: "4K UHD"
}

In [ ]:
mpx_type = laptop[['Screen_type', 'SR_weight', 'SR_height']]
mpx_type['mpx'] = (mpx_type['SR_weight'] * mpx_type['SR_height'] / 1000000).round(3)
mpx_type['mpx'].unique()

array([4.096, 1.296, 2.074, 5.184, 1.049, 3.318, 5.76 , 2.304, 3.393,
       8.294, 3.11 , 3.686, 1.44 , 4.99 , 3.84 ])

In [ ]:
std_type = {'Full HD': 'FHD', 'Quad HD': 'QHD', 'Ultra HD': '4K UHD'}
mpx_type['Screen_type_std'] = mpx_type['Screen_type'].replace(std_type)
mpx_type['Screen_type_std'] = mpx_type['mpx'].map(mpx_to_type)
mpx_type

,Screen_type,SR_weight,SR_height,mpx,Screen_type_std
0,NaN,2560,1600,4.096,WQXGA
1,NaN,1440,900,1.296,WXGA+
2,Full HD,1920,1080,2.074,FHD
3,NaN,2880,1800,5.184,NaN
4,NaN,2560,1600,4.096,WQXGA
...,...,...,...,...,...
1298,Full HD,1920,1080,2.074,FHD
1299,Quad HD,3200,1800,5.760,NaN
1300,NaN,1366,768,1.049,HD
1301,NaN,1366,768,1.049,HD


из интереса заодно глянем, совпадают ли типы с характеристиками

In [ ]:
mpx_type['Screen_type_std'] = mpx_type['Screen_type_std'].replace({v : k for k, v in std_type.items()})
mpx_type[['Screen_type', 'Screen_type_std', 'mpx']]

,Screen_type,Screen_type_std,mpx
0,NaN,WQXGA,4.096
1,NaN,WXGA+,1.296
2,Full HD,Full HD,2.074
3,NaN,NaN,5.184
4,NaN,WQXGA,4.096
...,...,...,...
1298,Full HD,Full HD,2.074
1299,Quad HD,NaN,5.760
1300,NaN,HD,1.049
1301,NaN,HD,1.049


In [ ]:
print(pd.__version__)

2.2.2


In [ ]:
check = mpx_type[mpx_type['Screen_type'].notna()]
check[check['Screen_type'] != check['Screen_type_std']]

,Screen_type,SR_weight,SR_height,mpx,Screen_type_std
33,Quad HD,3200,1800,5.760,NaN
111,Quad HD,3200,1800,5.760,NaN
170,Full HD,2160,1440,3.110,NaN
214,Full HD,2160,1440,3.110,NaN
323,Full HD,1366,768,1.049,HD
411,Full HD,2560,1440,3.686,Quad HD
540,Quad HD,3200,1800,5.760,NaN
542,Quad HD,3200,1800,5.760,NaN
562,Quad HD,3200,1800,5.760,NaN
636,Quad HD,3200,1800,5.760,NaN


In [ ]:
# много интересного, но на данный момент заполненные не трогаем, заполняем только пропуски
mpx_type.loc[mpx_type['Screen_type'].isna(), 'Screen_type'] = mpx_type.loc[mpx_type['Screen_type'].isna(), 'Screen_type_std']
mpx_type

,Screen_type,SR_weight,SR_height,mpx,Screen_type_std
0,WQXGA,2560,1600,4.096,WQXGA
1,WXGA+,1440,900,1.296,WXGA+
2,Full HD,1920,1080,2.074,Full HD
3,NaN,2880,1800,5.184,NaN
4,WQXGA,2560,1600,4.096,WQXGA
...,...,...,...,...,...
1298,Full HD,1920,1080,2.074,Full HD
1299,Quad HD,3200,1800,5.760,NaN
1300,HD,1366,768,1.049,HD
1301,HD,1366,768,1.049,HD


In [ ]:
mpx_type.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Screen_type      1282 non-null   object 
 1   SR_weight        1303 non-null   int64  
 2   SR_height        1303 non-null   int64  
 3   mpx              1303 non-null   float64
 4   Screen_type_std  1253 non-null   object 
dtypes: float64(1), int64(2), object(2)
memory usage: 51.0+ KB


In [ ]:
laptop['Screen_type'] = mpx_type['Screen_type']
laptop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           1303 non-null   object 
 1   TypeName          1303 non-null   object 
 2   Inches            1303 non-null   float64
 3   ScreenResolution  1303 non-null   object 
 4   Cpu               1303 non-null   object 
 5   Ram               1303 non-null   object 
 6   Memory            1303 non-null   object 
 7   Gpu               1303 non-null   object 
 8   OpSys             1303 non-null   object 
 9   Weight            1303 non-null   object 
 10  Price             1303 non-null   float64
 11  Screen_type       1282 non-null   object 
 12  SR_weight         1303 non-null   int64  
 13  SR_height         1303 non-null   int64  
 14  Touchscreen       1303 non-null   int64  
dtypes: float64(2), int64(3), object(10)
memory usage: 152.8+ KB


In [ ]:
laptop['Screen_type'] = laptop['Screen_type'].fillna('unknown')

## Признак `Cpu` разбить на 3:
- **Cpu_company**: Производитель (Intel, AMD и т.д.)
- **Cpu_name**: Имя процессора (Core i5, Core i5 7200U и т.д)
- **Cpu_GHz**: Тактовая частота (1.8GHz, 2.7GHz и т.д) GHz - убрать, признак должен быть числовым.

Подумать как это сделать.

In [ ]:
laptop.Cpu.unique()

array(['Intel Core i5 2.3GHz', 'Intel Core i5 1.8GHz',
       'Intel Core i5 7200U 2.5GHz', 'Intel Core i7 2.7GHz',
       'Intel Core i5 3.1GHz', 'AMD A9-Series 9420 3GHz',
       'Intel Core i7 2.2GHz', 'Intel Core i7 8550U 1.8GHz',
       'Intel Core i5 8250U 1.6GHz', 'Intel Core i3 6006U 2GHz',
       'Intel Core i7 2.8GHz', 'Intel Core M m3 1.2GHz',
       'Intel Core i7 7500U 2.7GHz', 'Intel Core i7 2.9GHz',
       'Intel Core i3 7100U 2.4GHz', 'Intel Atom x5-Z8350 1.44GHz',
       'Intel Core i5 7300HQ 2.5GHz', 'AMD E-Series E2-9000e 1.5GHz',
       'Intel Core i5 1.6GHz', 'Intel Core i7 8650U 1.9GHz',
       'Intel Atom x5-Z8300 1.44GHz', 'AMD E-Series E2-6110 1.5GHz',
       'AMD A6-Series 9220 2.5GHz',
       'Intel Celeron Dual Core N3350 1.1GHz',
       'Intel Core i3 7130U 2.7GHz', 'Intel Core i7 7700HQ 2.8GHz',
       'Intel Core i5 2.0GHz', 'AMD Ryzen 1700 3GHz',
       'Intel Pentium Quad Core N4200 1.1GHz',
       'Intel Atom x5-Z8550 1.44GHz',
       'Intel Celeron Du

In [ ]:
laptop[['Cpu_company', 'Cpu_name', 'Cpu_GHz']] = laptop['Cpu'].str.extract(r'^(\w+)?\s(.*?)\s(\d+\.?\d*)[Gh][Hh][Zz]*$')

In [ ]:
laptop['Cpu_company'].unique()

array(['Intel', 'AMD', 'Samsung'], dtype=object)

In [ ]:
laptop['Cpu_GHz'] = laptop['Cpu_GHz'].astype(float)
laptop['Cpu_GHz'].unique()

array([2.3 , 1.8 , 2.5 , 2.7 , 3.1 , 3.  , 2.2 , 1.6 , 2.  , 2.8 , 1.2 ,
       2.9 , 2.4 , 1.44, 1.5 , 1.9 , 1.1 , 1.3 , 2.6 , 3.6 , 3.2 , 1.  ,
       2.1 , 0.9 , 1.92])

In [ ]:
print(laptop['Cpu_name'].drop_duplicates().sort_values().to_list())

['A10-Series 9600P', 'A10-Series 9620P', 'A10-Series A10-9620P', 'A12-Series 9700P', 'A12-Series 9720P', 'A4-Series 7210', 'A6-Series 7310', 'A6-Series 9220', 'A6-Series A6-9220', 'A8-Series 7410', 'A9-Series 9410', 'A9-Series 9420', 'A9-Series A9-9420', 'Atom X5-Z8350', 'Atom Z8350', 'Atom x5-Z8300', 'Atom x5-Z8350', 'Atom x5-Z8550', 'Celeron Dual Core 3205U', 'Celeron Dual Core 3855U', 'Celeron Dual Core N3050', 'Celeron Dual Core N3060', 'Celeron Dual Core N3350', 'Celeron Quad Core N3160', 'Celeron Quad Core N3450', 'Celeron Quad Core N3710', 'Core M', 'Core M 6Y30', 'Core M 6Y54', 'Core M 6Y75', 'Core M 7Y30', 'Core M M3-6Y30', 'Core M M7-6Y75', 'Core M m3', 'Core M m3-7Y30', 'Core M m7-6Y75', 'Core i3 6006U', 'Core i3 6100U', 'Core i3 7100U', 'Core i3 7130U', 'Core i5', 'Core i5 6200U', 'Core i5 6260U', 'Core i5 6300HQ', 'Core i5 6300U', 'Core i5 6440HQ', 'Core i5 7200U', 'Core i5 7300HQ', 'Core i5 7300U', 'Core i5 7440HQ', 'Core i5 7500U', 'Core i5 7Y54', 'Core i5 7Y57', 'Core i

### чистка названий процессоров
надо чистить названия, сначала AMD (A_-Series)

#### AMD

In [ ]:
AMD = laptop[laptop['Cpu_company'] == 'AMD'].copy()
AMD['Cpu_name'].unique()

array(['A9-Series 9420', 'E-Series E2-9000e', 'E-Series E2-6110',
       'A6-Series 9220', 'Ryzen 1700', 'FX 9830P', 'E-Series 6110',
       'E-Series 9000e', 'A10-Series A10-9620P', 'A6-Series A6-9220',
       'A10-Series 9600P', 'A8-Series 7410', 'A12-Series 9720P',
       'Ryzen 1600', 'A10-Series 9620P', 'E-Series 7110',
       'A9-Series A9-9420', 'E-Series E2-9000', 'A6-Series 7310',
       'A12-Series 9700P', 'A4-Series 7210', 'FX 8800P', 'E-Series 9000',
       'A9-Series 9410'], dtype=object)

A6-Series 9220, A10-Series A10-9620P и т.п. - сходство налицо, это одна модель

In [ ]:
AMD[AMD['Cpu_name'].str.contains('A\d+', na=False)]

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price,Screen_type,SR_weight,SR_height,Touchscreen,Cpu_company,Cpu_name,Cpu_GHz
5,Acer,Notebook,15.6,1366x768,AMD A9-Series 9420 3GHz,4GB,500GB HDD,AMD Radeon R5,Windows 10,2.1kg,21312.0000,HD,1366,768,0,AMD,A9-Series 9420,3.0
32,HP,Notebook,17.3,Full HD 1920x1080,AMD A6-Series 9220 2.5GHz,4GB,500GB HDD,AMD Radeon 530,Windows 10,2.71kg,23389.9200,Full HD,1920,1080,0,AMD,A6-Series 9220,2.5
52,HP,Notebook,15.6,Full HD 1920x1080,AMD A6-Series 9220 2.5GHz,4GB,256GB SSD,AMD Radeon R4 Graphics,Windows 10,1.86kg,21231.5472,Full HD,1920,1080,0,AMD,A6-Series 9220,2.5
83,Lenovo,Notebook,15.6,Full HD 1920x1080,AMD A6-Series 9220 2.5GHz,4GB,128GB SSD,AMD R4 Graphics,Windows 10,2.2kg,21258.7200,Full HD,1920,1080,0,AMD,A6-Series 9220,2.5
84,Acer,Notebook,15.6,1366x768,AMD A9-Series 9420 3GHz,4GB,1TB HDD,AMD Radeon R5,Windows 10,2.1kg,21045.6000,HD,1366,768,0,AMD,A9-Series 9420,3.0
124,Acer,Notebook,15.6,1366x768,AMD A9-Series 9420 3GHz,4GB,256GB SSD,AMD Radeon R5,Windows 10,2.1kg,24029.2800,HD,1366,768,0,AMD,A9-Series 9420,3.0
144,HP,Notebook,15.6,1366x768,AMD A6-Series 9220 2.5GHz,4GB,256GB SSD,AMD Radeon R4 Graphics,Windows 10,1.86kg,19660.3200,HD,1366,768,0,AMD,A6-Series 9220,2.5
160,Asus,Notebook,15.6,1366x768,AMD A9-Series 9420 3GHz,4GB,1TB HDD,AMD Radeon R5 M420,Windows 10,2.03kg,21471.8400,HD,1366,768,0,AMD,A9-Series 9420,3.0
172,Lenovo,Notebook,15.6,1366x768,AMD A6-Series 9220 2.9GHz,4GB,500GB HDD,AMD Radeon R4 Graphics,No OS,2.2kg,16463.5200,HD,1366,768,0,AMD,A6-Series 9220,2.9
237,Asus,Notebook,15.6,Full HD 1920x1080,AMD A10-Series A10-9620P 2.5GHz,4GB,128GB SSD + 1TB HDD,AMD Radeon R5 M430,Windows 10,2.2kg,30636.0000,Full HD,1920,1080,0,AMD,A10-Series A10-9620P,2.5


In [ ]:
AMD['Cpu_name'] = AMD['Cpu_name'].replace(r'^([A-Z]\d+)-Series\s+\1-', r'\1-', regex=True)
print(AMD[AMD['Cpu_company'] == 'AMD']['Cpu_name'].drop_duplicates().sort_values().to_list())

['A10-9620P', 'A10-Series 9600P', 'A10-Series 9620P', 'A12-Series 9700P', 'A12-Series 9720P', 'A4-Series 7210', 'A6-9220', 'A6-Series 7310', 'A6-Series 9220', 'A8-Series 7410', 'A9-9420', 'A9-Series 9410', 'A9-Series 9420', 'E-Series 6110', 'E-Series 7110', 'E-Series 9000', 'E-Series 9000e', 'E-Series E2-6110', 'E-Series E2-9000', 'E-Series E2-9000e', 'FX 8800P', 'FX 9830P', 'Ryzen 1600', 'Ryzen 1700']


In [ ]:
AMD['Cpu_name'] = AMD['Cpu_name'].replace(r'^([A-Z]\d+)-Series\s+(\d+[A-Z]*\d*)', r'\1-\2', regex=True)
AMD['Cpu_name'].str.strip()
print(AMD[AMD['Cpu_company'] == 'AMD']['Cpu_name'].drop_duplicates().sort_values().to_list())

['A10-9600P', 'A10-9620P', 'A12-9700P', 'A12-9720P', 'A4-7210', 'A6-7310', 'A6-9220', 'A8-7410', 'A9-9410', 'A9-9420', 'E-Series 6110', 'E-Series 7110', 'E-Series 9000', 'E-Series 9000e', 'E-Series E2-6110', 'E-Series E2-9000', 'E-Series E2-9000e', 'FX 8800P', 'FX 9830P', 'Ryzen 1600', 'Ryzen 1700']


есть подозрение, что E-Series 9000 и E-Series E2-9000 - одно и то же. Врооооде интернет согласен, но проверим по частотам процессоров

In [ ]:
AMD[AMD['Cpu_name'].str.contains('^E-', na=False)][['Cpu', 'Cpu_name', 'Cpu_GHz']]

,Cpu,Cpu_name,Cpu_GHz
22,AMD E-Series E2-9000e 1.5GHz,E-Series E2-9000e,1.5
31,AMD E-Series E2-6110 1.5GHz,E-Series E2-6110,1.5
101,AMD E-Series E2-9000e 1.5GHz,E-Series E2-9000e,1.5
127,AMD E-Series 6110 1.5GHz,E-Series 6110,1.5
231,AMD E-Series 9000e 1.5GHz,E-Series 9000e,1.5
544,AMD E-Series 7110 1.8GHz,E-Series 7110,1.8
673,AMD E-Series E2-9000 2.2GHz,E-Series E2-9000,2.2
1151,AMD E-Series 7110 1.8GHz,E-Series 7110,1.8
1207,AMD E-Series 9000 2.2GHz,E-Series 9000,2.2


In [ ]:
AMD['Cpu_name'] = AMD['Cpu_name'].replace(r'^E-Series\s+(?:E2-)?(\d+[a-z]*)', r'E2-\1', regex=True)
AMD[AMD['Cpu_name'].str.contains('^E', na=False)][['Cpu', 'Cpu_name', 'Cpu_GHz']]

,Cpu,Cpu_name,Cpu_GHz
22,AMD E-Series E2-9000e 1.5GHz,E2-9000e,1.5
31,AMD E-Series E2-6110 1.5GHz,E2-6110,1.5
101,AMD E-Series E2-9000e 1.5GHz,E2-9000e,1.5
127,AMD E-Series 6110 1.5GHz,E2-6110,1.5
231,AMD E-Series 9000e 1.5GHz,E2-9000e,1.5
544,AMD E-Series 7110 1.8GHz,E2-7110,1.8
673,AMD E-Series E2-9000 2.2GHz,E2-9000,2.2
1151,AMD E-Series 7110 1.8GHz,E2-7110,1.8
1207,AMD E-Series 9000 2.2GHz,E2-9000,2.2


Вообще есть проблема, что процессоры, принадлежащие одной линейке, имеют разную частоту. Как показал поиск, в некоторых строках указана базовая или средняя частота процессора, а в некоторых максимальная. Поставим везде базовую частоту, но после того, как разберёмся с названиями всех моделей

In [ ]:
cpu_freqs = laptop.groupby('Cpu_name')['Cpu_GHz'].apply(list).reset_index()
cpu_freqs.columns = ['Cpu_name', 'all_GHz']
cpu_freqs

,Cpu_name,all_GHz
0,A10-Series 9600P,"[2.4, 2.4]"
1,A10-Series 9620P,"[2.5, 2.5]"
2,A10-Series A10-9620P,"[2.5, 2.5]"
3,A12-Series 9700P,[2.5]
4,A12-Series 9720P,"[2.7, 3.6, 3.6, 3.6, 3.6, 2.7, 3.6]"
...,...,...
88,Ryzen 1600,[3.2]
89,Ryzen 1700,"[3.0, 3.0, 3.0]"
90,Xeon E3-1505M V6,"[3.0, 3.0]"
91,Xeon E3-1535M v5,[2.9]


In [ ]:
# наглядный пример - A12-9720P
AMD[AMD['Cpu'].str.contains('9720P', na=False)][['Cpu', 'Cpu_name', 'Cpu_GHz']]

,Cpu,Cpu_name,Cpu_GHz
324,AMD A12-Series 9720P 2.7GHz,A12-9720P,2.7
341,AMD A12-Series 9720P 3.6GHz,A12-9720P,3.6
591,AMD A12-Series 9720P 3.6GHz,A12-9720P,3.6
702,AMD A12-Series 9720P 3.6GHz,A12-9720P,3.6
982,AMD A12-Series 9720P 3.6GHz,A12-9720P,3.6
1222,AMD A12-Series 9720P 2.7GHz,A12-9720P,2.7
1240,AMD A12-Series 9720P 3.6GHz,A12-9720P,3.6


#### Intel

In [ ]:
Intel = laptop[laptop['Cpu_company'] == 'Intel'].copy()
print(Intel['Cpu_name'].drop_duplicates().sort_values().to_list())
# есть случаи, когда указано только семейство процессора и частота

['Atom X5-Z8350', 'Atom Z8350', 'Atom x5-Z8300', 'Atom x5-Z8350', 'Atom x5-Z8550', 'Celeron Dual Core 3205U', 'Celeron Dual Core 3855U', 'Celeron Dual Core N3050', 'Celeron Dual Core N3060', 'Celeron Dual Core N3350', 'Celeron Quad Core N3160', 'Celeron Quad Core N3450', 'Celeron Quad Core N3710', 'Core M', 'Core M 6Y30', 'Core M 6Y54', 'Core M 6Y75', 'Core M 7Y30', 'Core M M3-6Y30', 'Core M M7-6Y75', 'Core M m3', 'Core M m3-7Y30', 'Core M m7-6Y75', 'Core i3 6006U', 'Core i3 6100U', 'Core i3 7100U', 'Core i3 7130U', 'Core i5', 'Core i5 6200U', 'Core i5 6260U', 'Core i5 6300HQ', 'Core i5 6300U', 'Core i5 6440HQ', 'Core i5 7200U', 'Core i5 7300HQ', 'Core i5 7300U', 'Core i5 7440HQ', 'Core i5 7500U', 'Core i5 7Y54', 'Core i5 7Y57', 'Core i5 8250U', 'Core i7', 'Core i7 6500U', 'Core i7 6560U', 'Core i7 6600U', 'Core i7 6700HQ', 'Core i7 6820HK', 'Core i7 6820HQ', 'Core i7 6920HQ', 'Core i7 7500U', 'Core i7 7560U', 'Core i7 7600U', 'Core i7 7660U', 'Core i7 7700HQ', 'Core i7 7820HK', 'Core 

Снова проблемы, что у одной линейки разные названия:

- 'Atom X5-Z8350', 'Atom Z8350', 'Atom x5-Z8350'
- Core M M3-6Y30, Core M 6Y30
- Core M 6Y75, Core M M7-6Y75, Core M m7-6Y75
- и др

Проблемы с разным регистром и лишними буквами перед концом модели.

In [ ]:
Intel[Intel['Cpu'].str.contains('Z8350|6Y30|6Y75', na=False)][['Cpu', 'Cpu_name']].sort_values(by='Cpu_name')

,Cpu,Cpu_name
1120,Intel Atom X5-Z8350 1.44GHz,Atom X5-Z8350
1041,Intel Atom X5-Z8350 1.44GHz,Atom X5-Z8350
718,Intel Atom Z8350 1.92GHz,Atom Z8350
20,Intel Atom x5-Z8350 1.44GHz,Atom x5-Z8350
483,Intel Atom x5-Z8350 1.44GHz,Atom x5-Z8350
556,Intel Atom x5-Z8350 1.44GHz,Atom x5-Z8350
575,Intel Atom x5-Z8350 1.44GHz,Atom x5-Z8350
626,Intel Atom x5-Z8350 1.44GHz,Atom x5-Z8350
1261,Intel Core M 6Y30 0.9GHz,Core M 6Y30
1289,Intel Core M 6Y30 0.9GHz,Core M 6Y30


Quad и Dual процессоры вообще разные, но если буквы и цифры модели, указывающие на конкретный идентификатор, одинаковы, то это вероятно одна модель.

In [ ]:
Intel[Intel['Cpu'].str.contains('(Dual|Quad)-?\s*Core N4200', na=False)][['Cpu_name', 'Cpu_GHz']].drop_duplicates()

,Cpu_name,Cpu_GHz
49,Pentium Quad Core N4200,1.1
779,Pentium Dual Core N4200,1.1


С интеловскими процессорами пентиум, селерон и зеон вроде всё норм (кроме Quad/Dual Core N4200)

In [ ]:
Intel[Intel['Cpu'].str.contains('Pentium|Celeron|Xeon', na=False)]['Cpu_name'].sort_values().unique()

array(['Celeron Dual Core 3205U', 'Celeron Dual Core 3855U',
       'Celeron Dual Core N3050', 'Celeron Dual Core N3060',
       'Celeron Dual Core N3350', 'Celeron Quad Core N3160',
       'Celeron Quad Core N3450', 'Celeron Quad Core N3710',
       'Pentium Dual Core 4405U', 'Pentium Dual Core 4405Y',
       'Pentium Dual Core N4200', 'Pentium Quad Core N3700',
       'Pentium Quad Core N3710', 'Pentium Quad Core N4200',
       'Xeon E3-1505M V6', 'Xeon E3-1535M v5', 'Xeon E3-1535M v6'],
      dtype=object)

In [ ]:
#оч много нюансов, как чистить в зависимости от семейства, поэтому выносим его отдельно для удобства
lowered = Intel['Cpu_name'].str.lower()
Intel[['family', 'model']] = lowered.str.extract(r'^(core\s+\S+|\S+)\s*?(.*?)$', expand=False) # если не кор, то семейство состоит из 1 слова
Intel['family'].unique()

array(['core i5', 'core i7', 'core i3', 'core m', 'atom', 'celeron',
       'pentium', 'xeon'], dtype=object)

In [ ]:
Intel['model'].unique()

array(['', ' 7200u', ' 8550u', ' 8250u', ' 6006u', ' m3', ' 7500u',
       ' 7100u', ' x5-z8350', ' 7300hq', ' 8650u', ' x5-z8300',
       ' dual core n3350', ' 7130u', ' 7700hq', ' quad core n4200',
       ' x5-z8550', ' dual core n3060', ' 7560u', ' 6200u', ' 6y75',
       ' 6920hq', ' 7y54', ' 7820hk', ' e3-1505m v6', ' 6500u', ' 6600u',
       ' dual core 3205u', ' 7820hq', ' 7600u', ' dual core 3855u',
       ' quad core n3710', ' 7300u', ' quad core n3450', ' 6440hq',
       ' 6820hq', ' 7y75', ' 7440hq', ' 7660u', ' m3-7y30', ' 7y57',
       ' 6700hq', ' 6100u', ' 6820hk', ' 7y30', ' e3-1535m v6',
       ' quad core n3160', ' 6300u', ' dual core n3050', ' m3-6y30',
       ' 6300hq', ' z8350', ' e3-1535m v5', ' 6260u', ' dual core n4200',
       ' dual core 4405u', ' 6560u', ' m7-6y75', ' dual core 4405y',
       ' quad core n3700', ' 6y54', ' 6y30'], dtype=object)

In [ ]:
# xeon не трогаем, в остальных берём только букавы с цифрами после дефисов
Intel.loc[Intel['family'] != 'xeon', 'model'] = Intel[Intel['family'] != 'xeon']['model'].str.extract(
    r'([A-Za-z0-9]+)$', expand=False).fillna('')
Intel['model'].unique()

array(['', '7200u', '8550u', '8250u', '6006u', 'm3', '7500u', '7100u',
       'z8350', '7300hq', '8650u', 'z8300', 'n3350', '7130u', '7700hq',
       'n4200', 'z8550', 'n3060', '7560u', '6200u', '6y75', '6920hq',
       '7y54', '7820hk', ' e3-1505m v6', '6500u', '6600u', '3205u',
       '7820hq', '7600u', '3855u', 'n3710', '7300u', 'n3450', '6440hq',
       '6820hq', '7y75', '7440hq', '7660u', '7y30', '7y57', '6700hq',
       '6100u', '6820hk', ' e3-1535m v6', 'n3160', '6300u', 'n3050',
       '6y30', '6300hq', ' e3-1535m v5', '6260u', '4405u', '6560u',
       '4405y', 'n3700', '6y54'], dtype=object)

In [ ]:
Intel['model'] = Intel['model'].str.upper()
Intel['family'] = Intel['family'].str.title().str.replace(r'\bI(\d+)\b', r'i\1', regex=True)
Intel[['Cpu', 'Cpu_name', 'family', 'model']].tail()

,Cpu,Cpu_name,family,model
1298,Intel Core i7 6500U 2.5GHz,Core i7 6500U,Core i7,6500U
1299,Intel Core i7 6500U 2.5GHz,Core i7 6500U,Core i7,6500U
1300,Intel Celeron Dual Core N3050 1.6GHz,Celeron Dual Core N3050,Celeron,N3050
1301,Intel Core i7 6500U 2.5GHz,Core i7 6500U,Core i7,6500U
1302,Intel Celeron Dual Core N3050 1.6GHz,Celeron Dual Core N3050,Celeron,N3050


In [ ]:
no_model = Intel.loc[Intel['model'] == '', 'family'].unique()
no_model

array(['Core i5', 'Core i7', 'Core M'], dtype=object)

In [ ]:
Intel[['Cpu', 'Cpu_name', 'family', 'model']].head()

,Cpu,Cpu_name,family,model
0,Intel Core i5 2.3GHz,Core i5,Core i5,
1,Intel Core i5 1.8GHz,Core i5,Core i5,
2,Intel Core i5 7200U 2.5GHz,Core i5 7200U,Core i5,7200U
3,Intel Core i7 2.7GHz,Core i7,Core i7,
4,Intel Core i5 3.1GHz,Core i5,Core i5,


In [ ]:
Intel['Cpu_name'] = Intel['family'] + ' ' + Intel['model']
print(Intel['Cpu_name'].drop_duplicates().sort_values().to_list())

['Atom Z8300', 'Atom Z8350', 'Atom Z8550', 'Celeron 3205U', 'Celeron 3855U', 'Celeron N3050', 'Celeron N3060', 'Celeron N3160', 'Celeron N3350', 'Celeron N3450', 'Celeron N3710', 'Core M ', 'Core M 6Y30', 'Core M 6Y54', 'Core M 6Y75', 'Core M 7Y30', 'Core M M3', 'Core i3 6006U', 'Core i3 6100U', 'Core i3 7100U', 'Core i3 7130U', 'Core i5 ', 'Core i5 6200U', 'Core i5 6260U', 'Core i5 6300HQ', 'Core i5 6300U', 'Core i5 6440HQ', 'Core i5 7200U', 'Core i5 7300HQ', 'Core i5 7300U', 'Core i5 7440HQ', 'Core i5 7500U', 'Core i5 7Y54', 'Core i5 7Y57', 'Core i5 8250U', 'Core i7 ', 'Core i7 6500U', 'Core i7 6560U', 'Core i7 6600U', 'Core i7 6700HQ', 'Core i7 6820HK', 'Core i7 6820HQ', 'Core i7 6920HQ', 'Core i7 7500U', 'Core i7 7560U', 'Core i7 7600U', 'Core i7 7660U', 'Core i7 7700HQ', 'Core i7 7820HK', 'Core i7 7820HQ', 'Core i7 7Y75', 'Core i7 8550U', 'Core i7 8650U', 'Pentium 4405U', 'Pentium 4405Y', 'Pentium N3700', 'Pentium N3710', 'Pentium N4200', 'Xeon  E3-1505M V6', 'Xeon  E3-1535M V5', 

In [ ]:
Intel.drop(columns=['family', 'model'], inplace=True)
Intel[['Cpu', 'Cpu_name', 'Cpu_GHz']].head()

,Cpu,Cpu_name,Cpu_GHz
0,Intel Core i5 2.3GHz,Core i5,2.3
1,Intel Core i5 1.8GHz,Core i5,1.8
2,Intel Core i5 7200U 2.5GHz,Core i5 7200U,2.5
3,Intel Core i7 2.7GHz,Core i7,2.7
4,Intel Core i5 3.1GHz,Core i5,3.1


#### Samsung
УРА!!! ОН ОДИН

In [ ]:
Samsung = laptop[laptop['Cpu_company'] == 'Samsung']
Samsung['Cpu_name'].unique()

array(['Cortex A72&A53'], dtype=object)

### расстановка базовых частот
У процессоров, принадлежащих одной линейке, указаны разные частоты. Как показывал поиск, в некоторых строках указана базовая или средняя частота процессора, а в некоторых максимальная.

Объединим названия и поставим везде базовую (кроме процессоров intel, где было только семейство, без модели - no_model)

In [ ]:
laptop.loc[laptop['Cpu_company'] == 'Intel'] = Intel
laptop.loc[laptop['Cpu_company'] == 'AMD'] = AMD
laptop['Cpu_name'] = laptop['Cpu_name'].str.strip()

In [ ]:
cpu_freqs = laptop.groupby('Cpu_name')['Cpu_GHz'].unique().reset_index()
cpu_freqs

,Cpu_name,Cpu_GHz
0,A10-9600P,[2.4]
1,A10-9620P,[2.5]
2,A12-9700P,[2.5]
3,A12-9720P,"[2.7, 3.6]"
4,A4-7210,[2.2]
...,...,...
75,Ryzen 1600,[3.2]
76,Ryzen 1700,[3.0]
77,Xeon E3-1505M V6,[3.0]
78,Xeon E3-1535M V5,[2.9]


In [ ]:
cpu_freqs[cpu_freqs['Cpu_GHz'].apply(len) > 1]

,Cpu_name,Cpu_GHz
3,A12-9720P,"[2.7, 3.6]"
6,A6-9220,"[2.5, 2.9]"
9,A9-9420,"[3.0, 2.9]"
11,Atom Z8350,"[1.44, 1.92]"
18,Celeron N3350,"[1.1, 2.0]"
21,Core M,"[1.2, 1.1]"
25,Core M 7Y30,"[2.2, 1.0]"
27,Core i3 6006U,"[2.0, 2.2]"
28,Core i3 6100U,"[2.3, 2.1]"
31,Core i5,"[2.3, 1.8, 3.1, 1.6, 2.0, 1.3, 2.9]"


In [ ]:
no_model

array(['Core i5', 'Core i7', 'Core M'], dtype=object)

In [ ]:
laptop[laptop['Cpu_name'].isin(no_model)][['Cpu', 'Cpu_name', 'Cpu_GHz']].head()

,Cpu,Cpu_name,Cpu_GHz
0,Intel Core i5 2.3GHz,Core i5,2.3
1,Intel Core i5 1.8GHz,Core i5,1.8
3,Intel Core i7 2.7GHz,Core i7,2.7
4,Intel Core i5 3.1GHz,Core i5,3.1
6,Intel Core i7 2.2GHz,Core i7,2.2


In [ ]:
base_freq = laptop.groupby('Cpu_name')['Cpu_GHz'].min()
not_in_model = ~laptop['Cpu_name'].isin(no_model)
laptop.loc[not_in_model, 'Cpu_GHz'] = laptop.loc[not_in_model, 'Cpu_name'].map(base_freq)
laptop[['Cpu', 'Cpu_name', 'Cpu_GHz']]

,Cpu,Cpu_name,Cpu_GHz
0,Intel Core i5 2.3GHz,Core i5,2.3
1,Intel Core i5 1.8GHz,Core i5,1.8
2,Intel Core i5 7200U 2.5GHz,Core i5 7200U,2.5
3,Intel Core i7 2.7GHz,Core i7,2.7
4,Intel Core i5 3.1GHz,Core i5,3.1
...,...,...,...
1298,Intel Core i7 6500U 2.5GHz,Core i7 6500U,2.5
1299,Intel Core i7 6500U 2.5GHz,Core i7 6500U,2.5
1300,Intel Celeron Dual Core N3050 1.6GHz,Celeron N3050,1.6
1301,Intel Core i7 6500U 2.5GHz,Core i7 6500U,2.5


In [ ]:
check_cpu_freqs = laptop.groupby('Cpu_name')['Cpu_GHz'].unique().reset_index()
check_cpu_freqs[check_cpu_freqs['Cpu_GHz'].apply(len) > 1]

,Cpu_name,Cpu_GHz
21,Core M,"[1.2, 1.1]"
31,Core i5,"[2.3, 1.8, 3.1, 1.6, 2.0, 1.3, 2.9]"
45,Core i7,"[2.7, 2.2, 2.8, 2.9]"


## По аналогии обработать признаки:
- **Ram**: сделать числовым
- **Memory**: разбить на 5 признаков:
    - количество дисков (256GB SSD +  2TB HDD = 2 диска, 256GB SSD = 1 диск)
    - тип жёского диска: SSD/HDD/Flash Storage/Hybrid/...
    - объём каждого диска (256GB/1TB/...) (сделать числовым)

**Пример**

| disk_count | disk_type | disk_1_V |  disk_2_V |
|:----------:|:---------:|:--------:|:--------:|
| 1          | SSD       | 128      | 0        |
| 2          | SSD/HDD   | 256      | 1000     |

- **Gpu**: разбить на 2 признака gpu_company и gpu_name
- **Weight**:  сделать числовым

## Все категориальные признаки закодировать (сделать числовыми)

### RAM

In [ ]:
laptop['Ram'].unique()

array(['8GB', '16GB', '4GB', '2GB', '12GB', '6GB', '32GB', '24GB', '64GB'],
      dtype=object)

In [ ]:
laptop['Ram'] = laptop['Ram'].replace('\D', '', regex=True).astype(int)
laptop['Ram'].unique()

array([ 8, 16,  4,  2, 12,  6, 32, 24, 64])

### Memory

In [ ]:
laptop['Memory'].unique()

array(['128GB SSD', '128GB Flash Storage', '256GB SSD', '512GB SSD',
       '500GB HDD', '256GB Flash Storage', '1TB HDD',
       '32GB Flash Storage', '128GB SSD +  1TB HDD',
       '256GB SSD +  256GB SSD', '64GB Flash Storage',
       '256GB SSD +  1TB HDD', '256GB SSD +  2TB HDD', '32GB SSD',
       '2TB HDD', '64GB SSD', '1.0TB Hybrid', '512GB SSD +  1TB HDD',
       '1TB SSD', '256GB SSD +  500GB HDD', '128GB SSD +  2TB HDD',
       '512GB SSD +  512GB SSD', '16GB SSD', '16GB Flash Storage',
       '512GB SSD +  256GB SSD', '512GB SSD +  2TB HDD',
       '64GB Flash Storage +  1TB HDD', '180GB SSD', '1TB HDD +  1TB HDD',
       '32GB HDD', '1TB SSD +  1TB HDD', '512GB Flash Storage',
       '128GB HDD', '240GB SSD', '8GB SSD', '508GB Hybrid', '1.0TB HDD',
       '512GB SSD +  1.0TB Hybrid', '256GB SSD +  1.0TB Hybrid'],
      dtype=object)

In [ ]:
disks = laptop['Memory'].str.split('\s?+\s+')
disks = disks.explode().to_frame('disk')
disks

,disk
0,128GB SSD
1,128GB Flash Storage
2,256GB SSD
3,512GB SSD
4,256GB SSD
...,...
1298,128GB SSD
1299,512GB SSD
1300,64GB Flash Storage
1301,1TB HDD


In [ ]:
disks[['_V', 'unit', '_type']] = disks['disk'].str.extract(r'([\d\.]+)\s*(GB|TB)\s+(.+)')
disks['_type'] = disks['_type'].str.replace('\W', '', regex=True)
disks

,disk,_V,unit,_type
0,128GB SSD,128,GB,SSD
1,128GB Flash Storage,128,GB,FlashStorage
2,256GB SSD,256,GB,SSD
3,512GB SSD,512,GB,SSD
4,256GB SSD,256,GB,SSD
...,...,...,...,...
1298,128GB SSD,128,GB,SSD
1299,512GB SSD,512,GB,SSD
1300,64GB Flash Storage,64,GB,FlashStorage
1301,1TB HDD,1,TB,HDD


In [ ]:
disks['_V'] = disks['_V'].astype(float)
disks.loc[disks['unit'] == 'TB', '_V'] *= 1000
disks['_V'] = disks['_V'].astype(int)
disks.drop(columns='unit', inplace=True)
disks

,disk,_V,_type
0,128GB SSD,128,SSD
1,128GB Flash Storage,128,FlashStorage
2,256GB SSD,256,SSD
3,512GB SSD,512,SSD
4,256GB SSD,256,SSD
...,...,...,...
1298,128GB SSD,128,SSD
1299,512GB SSD,512,SSD
1300,64GB Flash Storage,64,FlashStorage
1301,1TB HDD,1000,HDD


In [ ]:
# номер диска каждого индекса
disks['_count'] = disks.groupby(level=0).cumcount() + 1
disks[28:]

,disk,_V,_type,_count
27,256GB SSD,256,SSD,1
28,256GB SSD +,256,SSD,1
28,256GB SSD,256,SSD,2
29,1TB HDD,1000,HDD,1
30,64GB Flash Storage,64,FlashStorage,1
...,...,...,...,...
1298,128GB SSD,128,SSD,1
1299,512GB SSD,512,SSD,1
1300,64GB Flash Storage,64,FlashStorage,1
1301,1TB HDD,1000,HDD,1


In [ ]:
pivot_V = disks.pivot(columns='_count', values='_V').fillna(0).astype(int)
pivot_type = disks.pivot(columns='_count', values='_type')
pivot_V.columns = [f'disk_{i}_V' for i in pivot_V.columns]
pivot_type['disk_type'] = pivot_type.apply(lambda row: '/'.join([str(t) for t in row if pd.notna(t)]), axis=1)

In [ ]:
pivot_V[['disk_1_V', 'disk_2_V']]

,disk_1_V,disk_2_V
0,128,0
1,128,0
2,256,0
3,512,0
4,256,0
...,...,...
1298,128,0
1299,512,0
1300,64,0
1301,1000,0


In [ ]:
laptop['disk_count'] = laptop['Memory'].str.split(' + ').str.len().fillna(0).astype(int)
laptop['disk_type'] = pivot_type['disk_type']
laptop[['disk_1_V', 'disk_2_V']] = pivot_V[['disk_1_V', 'disk_2_V']]
laptop

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,...,SR_weight,SR_height,Touchscreen,Cpu_company,Cpu_name,Cpu_GHz,disk_count,disk_type,disk_1_V,disk_2_V
0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,...,2560,1600,0,Intel,Core i5,2.3,1,SSD,128,0
1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,...,1440,900,0,Intel,Core i5,1.8,1,FlashStorage,128,0
2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,...,1920,1080,0,Intel,Core i5 7200U,2.5,1,SSD,256,0
3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,...,2880,1800,0,Intel,Core i7,2.7,1,SSD,512,0
4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,...,2560,1600,0,Intel,Core i5,3.1,1,SSD,256,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,Lenovo,2 in 1 Convertible,14.0,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 6500U 2.5GHz,4,128GB SSD,Intel HD Graphics 520,Windows 10,1.8kg,...,1920,1080,1,Intel,Core i7 6500U,2.5,1,SSD,128,0
1299,Lenovo,2 in 1 Convertible,13.3,IPS Panel Quad HD+ / Touchscreen 3200x1800,Intel Core i7 6500U 2.5GHz,16,512GB SSD,Intel HD Graphics 520,Windows 10,1.3kg,...,3200,1800,1,Intel,Core i7 6500U,2.5,1,SSD,512,0
1300,Lenovo,Notebook,14.0,1366x768,Intel Celeron Dual Core N3050 1.6GHz,2,64GB Flash Storage,Intel HD Graphics,Windows 10,1.5kg,...,1366,768,0,Intel,Celeron N3050,1.6,1,FlashStorage,64,0
1301,HP,Notebook,15.6,1366x768,Intel Core i7 6500U 2.5GHz,6,1TB HDD,AMD Radeon R5 M330,Windows 10,2.19kg,...,1366,768,0,Intel,Core i7 6500U,2.5,1,HDD,1000,0


### GPU

In [ ]:
laptop['Gpu'].unique()

array(['Intel Iris Plus Graphics 640', 'Intel HD Graphics 6000',
       'Intel HD Graphics 620', 'AMD Radeon Pro 455',
       'Intel Iris Plus Graphics 650', 'AMD Radeon R5',
       'Intel Iris Pro Graphics', 'Nvidia GeForce MX150',
       'Intel UHD Graphics 620', 'Intel HD Graphics 520',
       'AMD Radeon Pro 555', 'AMD Radeon R5 M430',
       'Intel HD Graphics 615', 'AMD Radeon Pro 560',
       'Nvidia GeForce 940MX', 'Intel HD Graphics 400',
       'Nvidia GeForce GTX 1050', 'AMD Radeon R2', 'AMD Radeon 530',
       'Nvidia GeForce 930MX', 'Intel HD Graphics',
       'Intel HD Graphics 500', 'Nvidia GeForce 930MX ',
       'Nvidia GeForce GTX 1060', 'Nvidia GeForce 150MX',
       'Intel Iris Graphics 540', 'AMD Radeon RX 580',
       'Nvidia GeForce 920MX', 'AMD Radeon R4 Graphics', 'AMD Radeon 520',
       'Nvidia GeForce GTX 1070', 'Nvidia GeForce GTX 1050 Ti',
       'Nvidia GeForce MX130', 'AMD R4 Graphics',
       'Nvidia GeForce GTX 940MX', 'AMD Radeon RX 560',
       'Nvid

In [ ]:
laptop['Gpu']

,Gpu
0,Intel Iris Plus Graphics 640
1,Intel HD Graphics 6000
2,Intel HD Graphics 620
3,AMD Radeon Pro 455
4,Intel Iris Plus Graphics 650
...,...
1298,Intel HD Graphics 520
1299,Intel HD Graphics 520
1300,Intel HD Graphics
1301,AMD Radeon R5 M330


In [ ]:
laptop[['gpu_company', 'gpu_name']] = laptop['Gpu'].str.extract(r'^(\S+)?\s*(.*)')
laptop['gpu_name'] = laptop['gpu_name'].str.strip()
laptop[['Gpu', 'gpu_company', 'gpu_name']]

,Gpu,gpu_company,gpu_name
0,Intel Iris Plus Graphics 640,Intel,Iris Plus Graphics 640
1,Intel HD Graphics 6000,Intel,HD Graphics 6000
2,Intel HD Graphics 620,Intel,HD Graphics 620
3,AMD Radeon Pro 455,AMD,Radeon Pro 455
4,Intel Iris Plus Graphics 650,Intel,Iris Plus Graphics 650
...,...,...,...
1298,Intel HD Graphics 520,Intel,HD Graphics 520
1299,Intel HD Graphics 520,Intel,HD Graphics 520
1300,Intel HD Graphics,Intel,HD Graphics
1301,AMD Radeon R5 M330,AMD,Radeon R5 M330


In [ ]:
laptop['gpu_company'].unique()

array(['Intel', 'AMD', 'Nvidia', 'ARM'], dtype=object)

#### Апяц. AMD

Удивительно, магия, но тут как будто вообще без дубликатов

In [ ]:
AMD = laptop[laptop['gpu_company'] == 'AMD']
AMD['gpu_name'].sort_values().unique()

array(['FirePro W4190M', 'FirePro W5130M', 'FirePro W6150M', 'R17M-M1-70',
       'R4 Graphics', 'Radeon 520', 'Radeon 530', 'Radeon 540',
       'Radeon Pro 455', 'Radeon Pro 555', 'Radeon Pro 560', 'Radeon R2',
       'Radeon R2 Graphics', 'Radeon R3', 'Radeon R4',
       'Radeon R4 Graphics', 'Radeon R5', 'Radeon R5 430',
       'Radeon R5 520', 'Radeon R5 M315', 'Radeon R5 M330',
       'Radeon R5 M420', 'Radeon R5 M420X', 'Radeon R5 M430', 'Radeon R7',
       'Radeon R7 Graphics', 'Radeon R7 M360', 'Radeon R7 M365X',
       'Radeon R7 M440', 'Radeon R7 M445', 'Radeon R7 M460',
       'Radeon R7 M465', 'Radeon R9 M385', 'Radeon RX 540',
       'Radeon RX 550', 'Radeon RX 560', 'Radeon RX 580'], dtype=object)

#### Intel
зато тут есть

- Graphics 620, UHD Graphics 620
- HD Graphics 540, Iris Graphics 540

In [ ]:
Intel = laptop[laptop['gpu_company'] == 'Intel'].copy()
Intel['gpu_name'].sort_values().unique()

array(['Graphics 620', 'HD Graphics', 'HD Graphics 400',
       'HD Graphics 405', 'HD Graphics 500', 'HD Graphics 505',
       'HD Graphics 510', 'HD Graphics 515', 'HD Graphics 520',
       'HD Graphics 530', 'HD Graphics 5300', 'HD Graphics 540',
       'HD Graphics 6000', 'HD Graphics 615', 'HD Graphics 620',
       'HD Graphics 630', 'Iris Graphics 540', 'Iris Graphics 550',
       'Iris Plus Graphics 640', 'Iris Plus Graphics 650',
       'Iris Pro Graphics', 'UHD Graphics 620'], dtype=object)

In [ ]:
Intel['gpu_name'] = Intel['gpu_name'].str.extract(r'(\w+\s+\d+)$')
# восстанавливать полностью удаленные значения
originals = laptop[laptop['gpu_company'] == 'Intel'].loc[Intel['gpu_name'].isna(), ['gpu_name']]
Intel.loc[Intel['gpu_name'].isna(), 'gpu_name'] = originals
Intel['gpu_name'].sort_values().unique()

array(['Graphics 400', 'Graphics 405', 'Graphics 500', 'Graphics 505',
       'Graphics 510', 'Graphics 515', 'Graphics 520', 'Graphics 530',
       'Graphics 5300', 'Graphics 540', 'Graphics 550', 'Graphics 6000',
       'Graphics 615', 'Graphics 620', 'Graphics 630', 'Graphics 640',
       'Graphics 650', 'HD Graphics', 'Iris Pro Graphics'], dtype=object)

#### NVIDIA

просто снова куча дублей разного формата Т_Т

- 'GeForce GTX 1050 Ti', 'GeForce GTX 1050Ti', 'GeForce GTX1050 Ti'
- 'GeForce 940MX', 'GeForce GT 940MX', 'GeForce GTX 940MX'

In [ ]:
Nvidia = laptop[laptop['gpu_company'] == 'Nvidia'].copy()
Nvidia['gpu_name'].sort_values().unique()

array(['GTX 980 SLI', 'GeForce 150MX', 'GeForce 920', 'GeForce 920M',
       'GeForce 920MX', 'GeForce 930M', 'GeForce 930MX', 'GeForce 940M',
       'GeForce 940MX', 'GeForce 960M', 'GeForce GT 940MX',
       'GeForce GTX 1050', 'GeForce GTX 1050 Ti', 'GeForce GTX 1050M',
       'GeForce GTX 1050Ti', 'GeForce GTX 1060', 'GeForce GTX 1070',
       'GeForce GTX 1070M', 'GeForce GTX 1080', 'GeForce GTX 930MX',
       'GeForce GTX 940M', 'GeForce GTX 940MX', 'GeForce GTX 950M',
       'GeForce GTX 960', 'GeForce GTX 960<U+039C>', 'GeForce GTX 960M',
       'GeForce GTX 965M', 'GeForce GTX 970M', 'GeForce GTX 980',
       'GeForce GTX 980M', 'GeForce GTX1050 Ti', 'GeForce GTX1060',
       'GeForce GTX1080', 'GeForce MX130', 'GeForce MX150',
       'Quadro 3000M', 'Quadro M1000M', 'Quadro M1200', 'Quadro M2000M',
       'Quadro M2200', 'Quadro M2200M', 'Quadro M3000M', 'Quadro M500M',
       'Quadro M520M', 'Quadro M620', 'Quadro M620M'], dtype=object)

In [ ]:
Nvidia['gpu_name'] = Nvidia['gpu_name'].str.replace(r'<[^>]*>', '', regex=True) # от GeForce GTX 960<U+039C>
Nvidia['gpu_name'] = Nvidia['gpu_name'].str.replace(r'\s+', '', regex=True) #без пробелов
Nvidia['gpu_name'] = Nvidia['gpu_name'].str.replace(r'(GTX)', r' \1', regex=True) # пробел перед GTX
Nvidia['gpu_name'] = Nvidia['gpu_name'].str.replace(r'(MX)', r' \1', regex=True) # пробел перед MX
Nvidia['gpu_name'] = Nvidia['gpu_name'].str.replace(r'(Ti)', r' \1', regex=True) # пробел перед Ti
Nvidia['gpu_name'] = Nvidia['gpu_name'].str.replace(r'(M)(?=\d|$)', r' \1', regex=True) # пробел перед M (если после цифра или конец)
Nvidia['gpu_name'] = Nvidia['gpu_name'].str.replace(r'(\d{3,4})(?=GTX|MX|Ti|M|$)', r'\1 ', regex=True) # пробел после числа
Nvidia['gpu_name'] = Nvidia['gpu_name'].str.strip()
Nvidia['gpu_name'].sort_values().unique()
# все ещё не до конца Т___Т

array(['GTX980SLI', 'GeForce GTX1050', 'GeForce GTX1050 M',
       'GeForce GTX1050 Ti', 'GeForce GTX1060', 'GeForce GTX1070',
       'GeForce GTX1070 M', 'GeForce GTX1080', 'GeForce GTX930 MX',
       'GeForce GTX940 M', 'GeForce GTX940 MX', 'GeForce GTX950 M',
       'GeForce GTX960', 'GeForce GTX960 M', 'GeForce GTX965 M',
       'GeForce GTX970 M', 'GeForce GTX980', 'GeForce GTX980 M',
       'GeForce MX130', 'GeForce MX150', 'GeForce150 MX', 'GeForce920',
       'GeForce920 M', 'GeForce920 MX', 'GeForce930 M', 'GeForce930 MX',
       'GeForce940 M', 'GeForce940 MX', 'GeForce960 M', 'GeForceGT940 MX',
       'Quadro M1000 M', 'Quadro M1200', 'Quadro M2000 M', 'Quadro M2200',
       'Quadro M2200 M', 'Quadro M3000 M', 'Quadro M500 M',
       'Quadro M520 M', 'Quadro M620', 'Quadro M620 M', 'Quadro3000 M'],
      dtype=object)

In [ ]:
print(Nvidia['gpu_name'].sort_values().drop_duplicates().to_list())

['GTX980SLI', 'GeForce GTX1050', 'GeForce GTX1050 M', 'GeForce GTX1050 Ti', 'GeForce GTX1060', 'GeForce GTX1070', 'GeForce GTX1070 M', 'GeForce GTX1080', 'GeForce GTX930 MX', 'GeForce GTX940 M', 'GeForce GTX940 MX', 'GeForce GTX950 M', 'GeForce GTX960', 'GeForce GTX960 M', 'GeForce GTX965 M', 'GeForce GTX970 M', 'GeForce GTX980', 'GeForce GTX980 M', 'GeForce MX130', 'GeForce MX150', 'GeForce150 MX', 'GeForce920', 'GeForce920 M', 'GeForce920 MX', 'GeForce930 M', 'GeForce930 MX', 'GeForce940 M', 'GeForce940 MX', 'GeForce960 M', 'GeForceGT940 MX', 'Quadro M1000 M', 'Quadro M1200', 'Quadro M2000 M', 'Quadro M2200', 'Quadro M2200 M', 'Quadro M3000 M', 'Quadro M500 M', 'Quadro M520 M', 'Quadro M620', 'Quadro M620 M', 'Quadro3000 M']


#### ARM
ура, просто 1 граф

In [ ]:
ARM = laptop[laptop['gpu_company'] == 'ARM']
ARM['gpu_name'].sort_values().unique()

array(['Mali T860 MP4'], dtype=object)

#### Подставить и объединить

In [ ]:
laptop.loc[laptop['gpu_company'] == 'Intel'] = Intel
laptop.loc[laptop['Cpu_company'] == 'AMD'] = AMD
laptop.loc[laptop['Cpu_company'] == 'Nvidia'] = Nvidia
laptop['Cpu_name'] = laptop['Cpu_name'].str.strip()

### Weight

In [ ]:
laptop['Weight'].sort_values().unique()

array(['0.69kg', '0.81kg', '0.91kg', '0.920kg', '0.92kg', '0.97kg',
       '0.98kg', '0.99kg', '1.05kg', '1.08kg', '1.09kg', '1.10kg',
       '1.11kg', '1.12kg', '1.13kg', '1.14kg', '1.15kg', '1.16kg',
       '1.17kg', '1.18kg', '1.19kg', '1.1kg', '1.21kg', '1.22kg',
       '1.23kg', '1.24kg', '1.252kg', '1.25kg', '1.26kg', '1.27kg',
       '1.28kg', '1.29kg', '1.2kg', '1.31kg', '1.32kg', '1.34kg',
       '1.35kg', '1.36kg', '1.37kg', '1.38kg', '1.39kg', '1.3kg',
       '1.41kg', '1.42kg', '1.43kg', '1.44kg', '1.45kg', '1.47kg',
       '1.48kg', '1.49kg', '1.4kg', '1.54kg', '1.55kg', '1.56kg',
       '1.58kg', '1.59kg', '1.5kg', '1.62kg', '1.63kg', '1.64kg',
       '1.65kg', '1.68kg', '1.6kg', '1.70kg', '1.71kg', '1.74kg',
       '1.75kg', '1.76kg', '1.78kg', '1.79kg', '1.7kg', '1.83kg',
       '1.84kg', '1.85kg', '1.86kg', '1.87kg', '1.88kg', '1.89kg',
       '1.8kg', '1.90kg', '1.91kg', '1.93kg', '1.94kg', '1.95kg',
       '1.96kg', '1.98kg', '1.99kg', '1.9kg', '2.02kg', '2.03kg',
  

In [ ]:
laptop['Weight'] = laptop['Weight'].replace(r'[A-Za-z]', '', regex=True).astype(float)
laptop['Weight'].sort_values().unique()

array([0.69 , 0.81 , 0.91 , 0.92 , 0.97 , 0.98 , 0.99 , 1.05 , 1.08 ,
       1.09 , 1.1  , 1.11 , 1.12 , 1.13 , 1.14 , 1.15 , 1.16 , 1.17 ,
       1.18 , 1.19 , 1.2  , 1.21 , 1.22 , 1.23 , 1.24 , 1.25 , 1.252,
       1.26 , 1.27 , 1.28 , 1.29 , 1.3  , 1.31 , 1.32 , 1.34 , 1.35 ,
       1.36 , 1.37 , 1.38 , 1.39 , 1.4  , 1.41 , 1.42 , 1.43 , 1.44 ,
       1.45 , 1.47 , 1.48 , 1.49 , 1.5  , 1.54 , 1.55 , 1.56 , 1.58 ,
       1.59 , 1.6  , 1.62 , 1.63 , 1.64 , 1.65 , 1.68 , 1.7  , 1.71 ,
       1.74 , 1.75 , 1.76 , 1.78 , 1.79 , 1.8  , 1.83 , 1.84 , 1.85 ,
       1.86 , 1.87 , 1.88 , 1.89 , 1.9  , 1.91 , 1.93 , 1.94 , 1.95 ,
       1.96 , 1.98 , 1.99 , 2.   , 2.02 , 2.03 , 2.04 , 2.05 , 2.06 ,
       2.07 , 2.08 , 2.09 , 2.1  , 2.13 , 2.14 , 2.15 , 2.16 , 2.17 ,
       2.18 , 2.19 , 2.191, 2.2  , 2.21 , 2.23 , 2.24 , 2.25 , 2.26 ,
       2.29 , 2.3  , 2.31 , 2.32 , 2.33 , 2.34 , 2.36 , 2.37 , 2.38 ,
       2.4  , 2.43 , 2.45 , 2.5  , 2.54 , 2.56 , 2.59 , 2.591, 2.6  ,
       2.62 , 2.63 ,

## ......

In [ ]:
laptop = laptop.drop(columns=['ScreenResolution', 'Cpu', 'Memory', 'Gpu'])

In [ ]:
laptop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 20 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Company      1303 non-null   object 
 1   TypeName     1303 non-null   object 
 2   Inches       1303 non-null   float64
 3   Ram          1303 non-null   int64  
 4   OpSys        1303 non-null   object 
 5   Weight       1303 non-null   float64
 6   Price        1303 non-null   float64
 7   Screen_type  1303 non-null   object 
 8   SR_weight    1303 non-null   int64  
 9   SR_height    1303 non-null   int64  
 10  Touchscreen  1303 non-null   int64  
 11  Cpu_company  1303 non-null   object 
 12  Cpu_name     1303 non-null   object 
 13  Cpu_GHz      1303 non-null   float64
 14  disk_count   1303 non-null   int64  
 15  disk_type    1303 non-null   object 
 16  disk_1_V     1303 non-null   int64  
 17  disk_2_V     1303 non-null   int64  
 18  gpu_company  1303 non-null   object 
 19  gpu_na

In [ ]:
for i in laptop.select_dtypes(include=['object']).columns:
    print('\t', i)
    print(laptop[i].sort_values().unique())
    print()

	 Company
['Acer' 'Apple' 'Asus' 'Chuwi' 'Dell' 'Fujitsu' 'Google' 'HP' 'Huawei'
 'LG' 'Lenovo' 'MSI' 'Mediacom' 'Microsoft' 'Razer' 'Samsung' 'Toshiba'
 'Vero' 'Xiaomi']

	 TypeName
['2 in 1 Convertible' 'Gaming' 'Netbook' 'Notebook' 'Ultrabook'
 'Workstation']

	 OpSys
['Android' 'Chrome OS' 'Linux' 'Mac OS X' 'No OS' 'Windows 10'
 'Windows 10 S' 'Windows 7' 'macOS']

	 Screen_type
['Full HD' 'HD' 'HD+' 'Quad HD' 'Ultra HD' 'WQXGA' 'WUXGA' 'WXGA+'
 'unknown']

	 Cpu_company
['AMD' 'Intel' 'Samsung']

	 Cpu_name
['A10-9600P' 'A10-9620P' 'A12-9700P' 'A12-9720P' 'A4-7210' 'A6-7310'
 'A6-9220' 'A8-7410' 'A9-9410' 'A9-9420' 'Atom Z8300' 'Atom Z8350'
 'Atom Z8550' 'Celeron 3205U' 'Celeron 3855U' 'Celeron N3050'
 'Celeron N3060' 'Celeron N3160' 'Celeron N3350' 'Celeron N3450'
 'Celeron N3710' 'Core M' 'Core M 6Y30' 'Core M 6Y54' 'Core M 6Y75'
 'Core M 7Y30' 'Core M M3' 'Core i3 6006U' 'Core i3 6100U' 'Core i3 7100U'
 'Core i3 7130U' 'Core i5' 'Core i5 6200U' 'Core i5 6260U'
 'Core i5 6300HQ

In [ ]:
data = laptop.copy()

## Выполнить анализ на наличие выбросов
- визуализировать (hist, hist+kde, boxplot scatterplot, pairplot)
- провести статистический анализ на выбросы (IQR, z-score)

In [ ]:
###

# Вопросы
1. Влияет ли кто производитель на стоимость ноутбука?
    - выделить ТОП-10 производителей и на их основе осуществить анализ.
2. Влияет ли тип ноутбука на его стоимость?
    - посчитать среднюю стоимость/медиану для каждого типа.
3. Влияет ли тип экрана и его широта и сысота на стоимость? а размер диагонали?
4. Есть ли разница в средней стоимости ноутбука в зависимости от того, что производитель CPU?
5. Влияет ли количество дисков на стоимость ноутбука? А их тип и объём?
6. Влияет ли размер RAM на настоимость?
7. Сильно ли большая разница между ноутбуками с установленной ОС и без неё?
8. Влияет ли вес на цену ноутбука?
____
Целевая переменная - какое у неё распределение?

In [ ]:
###

# ЗАДАЧА

> **Построить регрессионную модель любым известным вам способом. Целевая метрика $R$ - коэффициент детерминации**

Обязательно построить `baseline` - базовую модель (обычную линейную регрессию) на ваших данных с учтом требуемой в задании предобработки и посчитать на ней $R$, задача ваших дальнейших манипуляций - улучшить $R$, который вы получили на `baseline`

**Пояснения**:
- можете строить несколько разных моделей - главное продемонстрировать как улучшается ваше целевая метрика
- можете проводить несколько вычислительных экспериментов по подбору оптимальных параметров для одной модели, тоже демонстрируя как целевая мтерика улучшается
- можете поплотнее поработать с данными (например стандартизовать или ещё выполнить какие преобразования/генерацию признаков/отбор признаков и т.д.), тоже таким образом демонстрирую как целевая метрика улучшается.

In [ ]:
from sklearn.preprocessing import TargetEncoder

# One-Hot для низкокардинальных
data_enc = pd.get_dummies(data, columns=['TypeName', 'OpSys', 'Screen_type', 'Cpu_company', 'gpu_company'],
                     prefix=['TypeName', 'OpSys', 'Screen_type', 'Cpu_company', 'gpu_company'], dtype=int)
# Target для высококардинальных
te = TargetEncoder(smooth=1)
high_card = te.fit_transform(data[['Company', 'Cpu_name', 'gpu_name']], data['Price'])
data_enc[['Company', 'Cpu_name', 'gpu_name']] = high_card

# disk_type
disk_map = {'SSD/SSD': 3, 'FlashStorage': 2.5, 'SSD/Hybrid': 2.5, 'SSD': 2, 'SSD/HDD': 2,
            'Hybrid': 1.5, 'HDD': 1, 'FlashStorage/HDD': 1.5, 'HDD/HDD': 1}
data_enc['disk_type'] = data_enc['disk_type'].map(disk_map).fillna(1.5)

data_enc.head()

,Company,Inches,Ram,Weight,Price,SR_weight,SR_height,Touchscreen,Cpu_name,Cpu_GHz,...,Screen_type_WUXGA,Screen_type_WXGA+,Screen_type_unknown,Cpu_company_AMD,Cpu_company_Intel,Cpu_company_Samsung,gpu_company_AMD,gpu_company_ARM,gpu_company_Intel,gpu_company_Nvidia
0,89748.904735,13.3,8,1.37,71378.6832,2560,1600,0,77956.040965,2.3,...,0,0,0,0,1,0,0,0,1,0
1,89748.904735,13.3,8,1.34,47895.5232,1440,900,0,77956.040965,1.8,...,0,1,0,0,1,0,0,0,1,0
2,57244.131182,15.6,8,1.86,30636.0000,1920,1080,0,49238.479416,2.5,...,0,0,0,0,1,0,0,0,1,0
3,82430.674277,15.4,16,1.83,135195.3360,2880,1800,0,114212.986077,2.7,...,0,0,1,0,1,0,1,0,0,0
4,71675.387922,13.3,8,1.37,96095.8080,2560,1600,0,64056.302507,3.1,...,0,0,0,0,1,0,0,0,1,0


In [ ]:
data_enc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 46 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Company                      1303 non-null   float64
 1   Inches                       1303 non-null   float64
 2   Ram                          1303 non-null   int64  
 3   Weight                       1303 non-null   float64
 4   Price                        1303 non-null   float64
 5   SR_weight                    1303 non-null   int64  
 6   SR_height                    1303 non-null   int64  
 7   Touchscreen                  1303 non-null   int64  
 8   Cpu_name                     1303 non-null   float64
 9   Cpu_GHz                      1303 non-null   float64
 10  disk_count                   1303 non-null   int64  
 11  disk_type                    1303 non-null   float64
 12  disk_1_V                     1303 non-null   int64  
 13  disk_2_V          

In [ ]:
X = data_enc.drop(columns=['Price'])
y = data_enc['Price']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

# Полный пайплайн
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('regressor', LinearRegression())
])

# Кросс-валидация - т.к. датасет очень маленький
scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')
print(f"LinearRegression baseline: r2_score = {scores.mean():.4f}")

LinearRegression baseline: r2_score = 0.7708
